# Classificacao de Imagens com timm MobileNetV3

Este notebook classifica as imagens da pasta `images` usando o modelo `mobilenetv3_small_100.lamb_in1k` da biblioteca `timm`.

## Padrao de documentacao deste notebook
- Toda nova etapa deve ter **titulo** e **descricao** em celula markdown.
- Toda celula de codigo deve incluir **comentarios curtos** explicando a intencao.

## 1) Imports e configuracao inicial

Aqui importamos bibliotecas, definimos dispositivo de inferencia e configuramos caminhos de entrada.

In [1]:
# Importa bibliotecas basicas para sistema de arquivos e exibicao de resultados.
from pathlib import Path
from pprint import pprint

# Importa PyTorch e utilitarios de modelo/transforms da biblioteca timm.
import torch
from PIL import Image
import timm
from timm.data import resolve_model_data_config, create_transform, ImageNetInfo, infer_imagenet_subset

# Importa matplotlib para visualizacao dos resultados.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Define caminho da pasta de imagens e dispositivo de execucao.
images_dir = Path("images")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Exibe informacoes uteis de ambiente para depuracao.
print(f"timm: {timm.__version__}")
print(f"torch: {torch.__version__}")
print(f"device: {device}")

timm: 1.0.25
torch: 2.10.0
device: cpu


## 2) Carregamento do modelo e pre-processamento

Nesta etapa carregamos o modelo pre-treinado, montamos o transform correto e recuperamos os nomes das classes.

In [2]:
# Define identificador do modelo que sera utilizado para classificacao.
model_name = "mobilenetv3_small_100.lamb_in1k"

# Carrega o modelo com pesos pre-treinados e coloca em modo de inferencia.
model = timm.create_model(model_name, pretrained=True)
model.to(device)
model.eval()

# Resolve configuracao de dados do modelo e cria transform compativel.
data_config = resolve_model_data_config(model)
transform = create_transform(**data_config, is_training=False)

# Recupera os nomes das classes do subset ImageNet associado ao modelo.
imagenet_info = ImageNetInfo(infer_imagenet_subset(model))
label_names = imagenet_info.label_names()

print(f"Modelo carregado: {model_name}")
print(f"Numero de classes: {len(label_names)}")

model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

Modelo carregado: mobilenetv3_small_100.lamb_in1k
Numero de classes: 1000


## 3) Classificacao das imagens da pasta `images`

A celula abaixo percorre todas as imagens suportadas, roda inferencia e salva Top-1 e Top-5 para cada arquivo.

In [3]:
# Define extensoes de imagem permitidas para classificacao.
valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}

# Lista arquivos validos na pasta de entrada.
image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in valid_ext])
if not image_paths:
    raise FileNotFoundError("Nenhuma imagem encontrada na pasta 'images'.")

# Acumula resultados estruturados de classificacao por imagem.
results = []

with torch.inference_mode():
    for image_path in image_paths:
        # Abre imagem em RGB para garantir compatibilidade com o transform.
        image = Image.open(image_path).convert("RGB")

        # Aplica pre-processamento e adiciona dimensao de batch.
        input_tensor = transform(image).unsqueeze(0).to(device)

        # Executa inferencia e converte logits em probabilidades.
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)

        # Seleciona as 5 classes com maior probabilidade.
        top_probs, top_indices = torch.topk(probs, k=5, dim=1)
        top_probs = top_probs[0].cpu().tolist()
        top_indices = top_indices[0].cpu().tolist()

        # Monta lista com descricao legivel de cada classe (ex: "bulbul", "pizza").
        top5 = []
        for idx, prob in zip(top_indices, top_probs):
            top5.append({
                "class_id": int(idx),
                "label": imagenet_info.index_to_description(idx),
                "probability": float(prob),
            })

        # Guarda resultado final da imagem (Top-1 + Top-5) com caminho para exibicao.
        results.append({
            "image": image_path.name,
            "image_path": str(image_path),
            "top1": top5[0],
            "top5": top5,
        })

print(f"Total de imagens classificadas: {len(results)}")

Total de imagens classificadas: 4


## 4) Visualizacao dos resultados

Para cada imagem, exibe lado a lado: a imagem original e um grafico de barras horizontais com as Top-5 predicoes (eixo X = probabilidade, eixo Y = classe predita).

In [ ]:
for item in results:
    # Extrai nomes das classes e probabilidades do Top-5 (ordem invertida para o grafico).
    labels = [pred["label"] for pred in item["top5"]][::-1]
    probs = [pred["probability"] for pred in item["top5"]][::-1]

    # Cria figura com dois paineis: imagem a esquerda, grafico a direita.
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.8]})

    # Painel esquerdo: exibe a imagem original sem eixos.
    img = mpimg.imread(item["image_path"])
    ax_img.imshow(img)
    ax_img.set_title(item["image"], fontsize=12, fontweight="bold")
    ax_img.axis("off")

    # Painel direito: grafico de barras horizontais com Top-5 predicoes.
    colors = ["#2196F3"] * 4 + ["#4CAF50"]
    bars = ax_bar.barh(labels, probs, color=colors)
    ax_bar.set_xlabel("Probabilidade", fontsize=11)
    ax_bar.set_xlim(0, 1.0)
    ax_bar.set_title("Top-5 Predicoes", fontsize=12, fontweight="bold")

    # Adiciona o valor numerico ao lado de cada barra para leitura rapida.
    for bar, prob in zip(bars, probs):
        ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"{prob:.2%}", va="center", fontsize=10)

    plt.tight_layout()
    plt.show()

bird.jpg: n01560419 (0.1461)
cats_dog.png: n02087394 (0.1238)
kitchen.png: n03961711 (0.1120)
pizza.png: n07873807 (0.9676)

Resultados completos (Top-5 por imagem):
[{'image': 'bird.jpg',
  'top1': {'class_id': 16,
           'label': 'n01560419',
           'probability': 0.1460537165403366},
  'top5': [{'class_id': 16,
            'label': 'n01560419',
            'probability': 0.1460537165403366},
           {'class_id': 19,
            'label': 'n01592084',
            'probability': 0.1193181723356247},
           {'class_id': 13,
            'label': 'n01534433',
            'probability': 0.09974251687526703},
           {'class_id': 14,
            'label': 'n01537544',
            'probability': 0.048289284110069275},
           {'class_id': 17,
            'label': 'n01580077',
            'probability': 0.046005308628082275}]},
 {'image': 'cats_dog.png',
  'top1': {'class_id': 159,
           'label': 'n02087394',
           'probability': 0.12375235557556152},
  'top5': [